In [1]:
import copy
import torch
import torch.nn.functional as F
from torch.optim import AdamW

from utils.spectreTrack import SpectreTrack
from utils.dataset import GOT10kDataset

def get_freest_gpu():
    if not torch.cuda.is_available() or torch.cuda.device_count() == 0:
        return "cpu"
    
    # Get the ID of the GPU with the maximum free memory
    best_gpu = max(
        range(torch.cuda.device_count()), 
        key=lambda i: torch.cuda.mem_get_info(i)[0]
    )
    print(f"INFO:GPU {best_gpu} SELECTED")
    return f"cuda:{best_gpu}"

# Replace your original line with:
device = get_freest_gpu()
# device = "cuda:2"


def apply_adversarial_mask(
    search_clean,
    importance,
    patch_size=16,
    mask_ratio=0.20,
    mask_value="mean",
):
    """
    Mask the most salient search-image patches based on teacher search-token importance.

    Args:
        search_clean: [B, C, H, W], normalized search images.
        importance: [B, N], teacher search-token L2 norms.
        patch_size: ViT patch size.
        mask_ratio: fraction of search patches to mask.
        mask_value: "mean" uses 0 in normalized space, "black" uses true black after normalization.

    Returns:
        search_masked: [B, C, H, W]
    """
    B, C, H, W = search_clean.shape

    num_patches_h = H // patch_size
    num_patches_w = W // patch_size
    num_search_patches = num_patches_h * num_patches_w

    search_importance = importance[:, -num_search_patches:]

    num_mask = max(1, int(mask_ratio * num_search_patches))
    _, topk_indices = torch.topk(search_importance, num_mask, dim=-1)

    mask = torch.ones(
        (B, 1, num_patches_h, num_patches_w),
        device=search_clean.device,
        dtype=search_clean.dtype,
    )

    y_coords = topk_indices // num_patches_w
    x_coords = topk_indices % num_patches_w

    batch_indices = torch.arange(B, device=search_clean.device).view(B, 1)
    mask[batch_indices, 0, y_coords, x_coords] = 0.0

    mask_pixel = F.interpolate(
        mask,
        size=(H, W),
        mode="nearest",
    )

    if mask_value == "black":
        fill = torch.tensor(
            [-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
            device=search_clean.device,
            dtype=search_clean.dtype,
        ).view(1, 3, 1, 1)
    else:
        fill = torch.zeros(
            (1, C, 1, 1),
            device=search_clean.device,
            dtype=search_clean.dtype,
        )

    search_masked = search_clean * mask_pixel + fill * (1.0 - mask_pixel)

    return search_masked


@torch.no_grad()
def update_teacher_ema(student, teacher, decay=0.996):
    for teacher_param, student_param in zip(teacher.parameters(), student.parameters()):
        teacher_param.data.mul_(decay).add_(student_param.data, alpha=1.0 - decay)


spectre_net = SpectreTrack(
    model_name="vit_base_patch16_224.mae",
    pretrained=True,
).to(device)

teacher_net = copy.deepcopy(spectre_net).to(device)

for param in teacher_net.parameters():
    param.requires_grad = False

teacher_net.eval()

backbone_params = []
head_params = []

for name, param in spectre_net.named_parameters():
    if not param.requires_grad:
        continue

    if "backbone" in name:
        backbone_params.append(param)
    else:
        head_params.append(param)

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 3e-4},
    ],
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=40,
    gamma=0.1,
)

lambda_rec = 10.0

INFO:GPU 6 SELECTED
[INFO] Loading local checkpoint: /root/victor/vit_base_patch16_224.mae.pth
[WARNING] Local load failed: Checkpoint not found: /root/victor/vit_base_patch16_224.mae.pth
[INFO] Falling back to timm pretrained weights...


'[Errno 101] Network is unreachable' thrown while requesting HEAD https://huggingface.co/timm/vit_base_patch16_224.mae/resolve/main/model.safetensors
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [2]:
import torch
import importlib
import utils.dataset

importlib.reload(utils.dataset)

from utils.dataset import get_dataloader

data_path = '/root/victor/GOT-10K_FULL'

dataloader = get_dataloader(
    root_dir=data_path,
    batch_size=16,
    num_workers=8,
    template_size=128,
    search_size=256,
    max_frame_gap=20,
    samples_per_epoch=60000,
    translation_jitter=0.6,
    scale_jitter=0.20,
)

for template, search, bbox in dataloader:
    print(f"Template shape: {template.shape}")  # [16, 3, 128, 128]
    print(f"Search shape: {search.shape}")      # [16, 3, 256, 256]
    print(f"BBox shape: {bbox.shape}")          # [16, 4]
    print(f"BBox min/max: {bbox.min().item():.4f}, {bbox.max().item():.4f}")

    assert template.shape[-2:] == (128, 128)
    assert search.shape[-2:] == (256, 256)
    assert bbox.shape[-1] == 4
    assert torch.isfinite(template).all()
    assert torch.isfinite(search).all()
    assert torch.isfinite(bbox).all()
    assert bbox.min() >= 0.0 and bbox.max() <= 1.0

    break

Template shape: torch.Size([16, 3, 128, 128])
Search shape: torch.Size([16, 3, 256, 256])
BBox shape: torch.Size([16, 4])
BBox min/max: 0.1064, 0.8279


TRAINING CODE

In [ ]:
import torch
import torch.nn.functional as F
from torchvision.ops import generalized_box_iou


def cxcywh_to_xyxy(bbox):
    cx, cy, w, h = bbox.unbind(-1)

    w = w.clamp(min=1e-6)
    h = h.clamp(min=1e-6)

    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2

    return torch.stack([x1, y1, x2, y2], dim=-1).clamp(0.0, 1.0)


def calculate_giou_loss(pred_boxes, gt_boxes):
    pred_boxes_xyxy = cxcywh_to_xyxy(pred_boxes)
    gt_boxes_xyxy = cxcywh_to_xyxy(gt_boxes)

    giou_matrix = generalized_box_iou(pred_boxes_xyxy, gt_boxes_xyxy)
    giou = torch.diag(giou_matrix)

    return (1.0 - giou).mean()


@torch.no_grad()
def update_teacher_ema(student, teacher, momentum=0.999):
    for student_param, teacher_param in zip(student.parameters(), teacher.parameters()):
        teacher_param.data.mul_(momentum).add_(student_param.data, alpha=1.0 - momentum)


def train_epoch(
    dataloader,
    spectre_net,
    teacher_net,
    optimizer,
    device,
    epoch,
    lambda_rec=10.0,
    mask_ratio=0.20,
):
    spectre_net.train()
    teacher_net.eval()

    total_loss_epoch = 0.0

    for batch_idx, (template, search_clean, bbox_gt) in enumerate(dataloader):
        template = template.to(device, non_blocking=True)
        search_clean = search_clean.to(device, non_blocking=True)
        bbox_gt = bbox_gt.to(device, non_blocking=True)

        with torch.no_grad():
            features_teacher = teacher_net.extract_clean_features(template, search_clean)
            importance = torch.norm(features_teacher, dim=-1)

        search_masked = apply_adversarial_mask(
            search_clean,
            importance,
            patch_size=16,
            mask_ratio=mask_ratio,
        )

        optimizer.zero_grad(set_to_none=True)

        bbox_pred, features_student, confidence = spectre_net(
            template,
            search_masked,
            history_buffer=None,
        )

        loss_giou = calculate_giou_loss(bbox_pred, bbox_gt)
        loss_l1 = F.smooth_l1_loss(bbox_pred, bbox_gt)

        # Optional: all current samples are valid positives.
        confidence_target = torch.ones_like(confidence)
        loss_score = F.binary_cross_entropy(confidence, confidence_target)

        norm_student = F.normalize(features_student, p=2, dim=-1)
        norm_teacher = F.normalize(features_teacher.detach(), p=2, dim=-1)
        loss_rec = F.mse_loss(norm_student, norm_teacher)

        loss_loc = loss_giou + 2.0 * loss_l1 + 0.1 * loss_score
        total_loss = loss_loc + lambda_rec * loss_rec

        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(spectre_net.parameters(), max_norm=1.0)
        optimizer.step()

        update_teacher_ema(spectre_net, teacher_net, momentum=0.999)

        total_loss_epoch += total_loss.item()

        if batch_idx % 50 == 0:
            print(
                f"Epoch [{epoch}] Batch [{batch_idx}] | "
                f"GIoU: {loss_giou.item():.4f} | "
                f"L1: {loss_l1.item():.4f} | "
                f"Score: {loss_score.item():.4f} | "
                f"Rec: {loss_rec.item():.4f} | "
                f"Total: {total_loss.item():.4f}"
            )

    return total_loss_epoch / len(dataloader)


num_epochs = 50

print("Starting SpectreTrack GOT-10K fine-tuning...")
print("Using clean bbox labels, adversarial search masking, and EMA teacher.")

for epoch in range(num_epochs):
    avg_loss = train_epoch(
        dataloader=dataloader,
        spectre_net=spectre_net,
        teacher_net=teacher_net,
        optimizer=optimizer,
        device=device,
        epoch=epoch,
        lambda_rec=1.0,
        mask_ratio=0.20,
    )

    scheduler.step()

    print(f"--- Epoch {epoch} Completed | Average Loss: {avg_loss:.4f} ---")

    if epoch % 10 == 0 or epoch == num_epochs - 1:
        torch.save(
            spectre_net.state_dict(),
            f"spectre_got10k_mae_epoch_{epoch}.pth",
        )

EVALUATION ON VAL CODE

In [ ]:
import os
import glob
import torch
import numpy as np
import cv2
from tqdm import tqdm
from torchvision import transforms

# --- IMPORT YOUR REVERTED MODEL ---
from utils.spectreTrack import SpectreTrack

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------------------------------------------------
# 1. METRIC FUNCTIONS (Official GOT-10k)
# ---------------------------------------------------------
def calculate_iou(boxA, boxB):
    ax1, ay1, ax2, ay2 = boxA[0], boxA[1], boxA[0] + boxA[2], boxA[1] + boxA[3]
    bx1, by1, bx2, by2 = boxB[0], boxB[1], boxB[0] + boxB[2], boxB[1] + boxB[3]
    
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1) 
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    
    inter_w = max(0, ix2 - ix1)
    inter_h = max(0, iy2 - iy1)
    intersection = inter_w * inter_h
    
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    union = areaA + areaB - intersection
    
    return intersection / max(1e-6, union)

def calculate_success_rate(ious, threshold=0.5):
    return np.sum(np.array(ious) > threshold) / len(ious)

def calculate_ao(ious):
    return np.mean(ious)

# ---------------------------------------------------------
# 2. CROPPING & BOX UTILITIES
# ---------------------------------------------------------
def dataset_style_crop(image, bbox, target_size, context):
    h, w, _ = image.shape
    x, y, bw, bh = bbox
    cx, cy = x + bw / 2, y + bh / 2
    
    # Boundary clamping
    cx = np.clip(cx, 0, w - 1)
    cy = np.clip(cy, 0, h - 1)
    
    s_z = np.sqrt((bw + context * (bw + bh)) * (bh + context * (bw + bh)))
    max_sz = np.sqrt(w**2 + h**2) * 2
    s_z = min(max_sz, max(10.0, s_z))
    
    x1 = int(np.round(cx - s_z / 2))
    y1 = int(np.round(cy - s_z / 2))
    pad_x1, pad_y1 = max(0, -x1), max(0, -y1)
    
    image_padded = cv2.copyMakeBorder(
        image, pad_y1, max(0, int(cy + s_z/2) - h),
        pad_x1, max(0, int(cx + s_z/2) - w),
        cv2.BORDER_CONSTANT, value=(127, 127, 127)
    )
    
    crop = image_padded[y1+pad_y1:y1+pad_y1+int(s_z), x1+pad_x1:x1+pad_x1+int(s_z), :]
    crop_resized = cv2.resize(crop, (target_size, target_size))
    
    return crop_resized, {'x1': x1, 'y1': y1, 's_z': s_z}

def map_box_back_to_pixels(norm_box, crop_info, img_shape):
    cx_n, cy_n, w_n, h_n = norm_box
    w_n = np.clip(w_n, 0.01, 1.0)
    h_n = np.clip(h_n, 0.01, 1.0)
    
    s_z = crop_info['s_z']
    cx_c = cx_n * s_z
    cy_c = cy_n * s_z
    w_c = w_n * s_z
    h_c = h_n * s_z
    
    abs_x = cx_c + crop_info['x1'] - (w_c / 2)
    abs_y = cy_c + crop_info['y1'] - (h_c / 2)
    
    img_h, img_w = img_shape[:2]
    w_c = min(w_c, img_w)
    h_c = min(h_c, img_h)
    
    return np.array([abs_x, abs_y, w_c, h_c])

# ---------------------------------------------------------
# 3. TRACKING FUNCTION
# ---------------------------------------------------------
@torch.no_grad()
def track_sequence(seq_dir, spectre_net, transform, device):
    spectre_net.eval()
    
    gt_path = os.path.join(seq_dir, 'groundtruth.txt')
    gt_bboxes = np.loadtxt(gt_path, delimiter=',', ndmin=2)
    img_paths = sorted(glob.glob(os.path.join(seq_dir, '*.jpg')))
    
    predictions = []
    seq_ious = []
    
    # --- FIXED INITIALIZATION ---
    first_img_bgr = cv2.imread(img_paths[0])
    first_img_rgb = cv2.cvtColor(first_img_bgr, cv2.COLOR_BGR2RGB)
    first_bbox = gt_bboxes[0].astype(np.float32)
    
    # Create 128x128 template and 256x256 init-search for token alignment
    crop_template, _ = dataset_style_crop(first_img_rgb, first_bbox, target_size=128, context=0.5)
    tensor_template = transform(crop_template).unsqueeze(0).to(device)
    
    crop_search_init, _ = dataset_style_crop(first_img_rgb, first_bbox, target_size=256, context=2.0)
    tensor_search_init = transform(crop_search_init).unsqueeze(0).to(device)
    
    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        identity_buffer = spectre_net.extract_clean_features(tensor_template, tensor_search_init)
    
    state_bbox = first_bbox.copy()
    predictions.append(f"{state_bbox[0]:.1f},{state_bbox[1]:.1f},{state_bbox[2]:.1f},{state_bbox[3]:.1f}")
    
    for frame_idx in range(1, len(img_paths)):
        img_bgr = cv2.imread(img_paths[frame_idx])
        if img_bgr is None: continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_h, img_w = img_rgb.shape[:2]
        
        crop_search, crop_info = dataset_style_crop(img_rgb, state_bbox, target_size=256, context=2.0)
        tensor_search = transform(crop_search).unsqueeze(0).to(device)
        
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            pred_bbox_normalized, _, score_pred = spectre_net(
                tensor_template, tensor_search, history_buffer=identity_buffer
            )
            
        confidence = score_pred.item()
        if confidence < 0.2:
            predictions.append(f"{state_bbox[0]:.1f},{state_bbox[1]:.1f},{state_bbox[2]:.1f},{state_bbox[3]:.1f}")
            if frame_idx < len(gt_bboxes): seq_ious.append(calculate_iou(state_bbox, gt_bboxes[frame_idx]))
            continue

        pred_box_px = map_box_back_to_pixels(pred_bbox_normalized.squeeze(0).cpu().numpy(), crop_info, img_rgb.shape)
        
        old_x, old_y, old_w, old_h = state_bbox
        new_x, new_y, new_w, new_h = pred_box_px
        
        scale_lr = np.clip(confidence * 0.4, 0.05, 0.6)
        smoothed_w = (1.0 - scale_lr) * old_w + scale_lr * new_w
        smoothed_h = (1.0 - scale_lr) * old_h + scale_lr * new_h
        final_x = (new_x + new_w / 2) - (smoothed_w / 2)
        final_y = (new_y + new_h / 2) - (smoothed_h / 2)
        
        state_bbox = np.array([np.clip(final_x, 0, img_w - 1), np.clip(final_y, 0, img_h - 1), 
                               np.clip(smoothed_w, 5, img_w), np.clip(smoothed_h, 5, img_h)])
        
        if confidence > 0.6:
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                with torch.no_grad():
                    new_mem = spectre_net.extract_clean_features(tensor_template, tensor_search)
                identity_buffer = (1.0 - confidence * 0.05) * identity_buffer + (confidence * 0.05) * new_mem
            
        predictions.append(f"{state_bbox[0]:.1f},{state_bbox[1]:.1f},{state_bbox[2]:.1f},{state_bbox[3]:.1f}")
        if frame_idx < len(gt_bboxes): seq_ious.append(calculate_iou(state_bbox, gt_bboxes[frame_idx]))
    
    return predictions, seq_ious

# ---------------------------------------------------------
# 4. MAIN PIPELINE
# ---------------------------------------------------------
if __name__ == "__main__":
    val_data_path = '/root/victor/GOT-10K_FULL/val'
    val_sequences = sorted(glob.glob(os.path.join(val_data_path, 'GOT-10k_Val_*')))
    weights_path = 'spectre_got10k_mae_epoch_99.pth'
    results_dir = 'evaluation_results'
    os.makedirs(results_dir, exist_ok=True)
    
    spectre_net = SpectreTrack(model_name='vit_base_patch16_224.mae', pretrained=False)
    spectre_net.load_state_dict(torch.load(weights_path, map_location=device))
    spectre_net.to(device)
    
    img_transform = transforms.Compose([transforms.ToTensor(), 
                                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
    
    all_ious = []
    for seq_dir in tqdm(val_sequences, desc="Sequences"):
        seq_name = os.path.basename(seq_dir)
        predictions, seq_ious = track_sequence(seq_dir, spectre_net, img_transform, device)
        all_ious.extend(seq_ious)
        with open(os.path.join(results_dir, f"{seq_name}.txt"), 'w') as f:
            f.write('\n'.join(predictions))
            
    print(f"\nAO: {calculate_ao(all_ious)*100:.2f}% | SR@0.5: {calculate_success_rate(all_ious, 0.5)*100:.2f}%")

OFFICIAL TEST CODE

In [3]:
import os
import glob
import time
import zipfile
import torch
import numpy as np
import cv2

from tqdm import tqdm
from torchvision import transforms
from utils.spectreTrack import SpectreTrack


device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

test_data_path = "/root/victor/GOT-10K_FULL/test"
weights_path = "spectre_got10k_mae_epoch_99.pth"

tracker_name = "SpectreTrack"
results_root = "results"
num_repetitions = 3


def crop_search_region(image, bbox, output_size, context_amount):
    img_h, img_w = image.shape[:2]
    x, y, bw, bh = bbox.astype(np.float32)

    cx = x + bw * 0.5
    cy = y + bh * 0.5

    crop_size = np.sqrt(
        (bw + context_amount * (bw + bh)) *
        (bh + context_amount * (bw + bh))
    )
    crop_size = max(10.0, float(crop_size))

    x1 = int(np.floor(cx - crop_size * 0.5))
    y1 = int(np.floor(cy - crop_size * 0.5))
    x2 = int(np.ceil(cx + crop_size * 0.5))
    y2 = int(np.ceil(cy + crop_size * 0.5))

    pad_left = max(0, -x1)
    pad_top = max(0, -y1)
    pad_right = max(0, x2 - img_w)
    pad_bottom = max(0, y2 - img_h)

    if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
        image = cv2.copyMakeBorder(
            image,
            pad_top,
            pad_bottom,
            pad_left,
            pad_right,
            cv2.BORDER_CONSTANT,
            value=(127, 127, 127),
        )

    crop = image[
        y1 + pad_top:y2 + pad_top,
        x1 + pad_left:x2 + pad_left,
        :
    ]

    crop = cv2.resize(crop, (output_size, output_size), interpolation=cv2.INTER_LINEAR)

    crop_info = {
        "x1": x1,
        "y1": y1,
        "crop_w": float(x2 - x1),
        "crop_h": float(y2 - y1),
    }

    return crop, crop_info


def map_box_back(norm_box, crop_info, image_shape):
    cx, cy, bw, bh = norm_box.astype(np.float32)

    cx = np.clip(cx, 0.0, 1.0)
    cy = np.clip(cy, 0.0, 1.0)
    bw = np.clip(bw, 0.01, 1.0)
    bh = np.clip(bh, 0.01, 1.0)

    crop_w = crop_info["crop_w"]
    crop_h = crop_info["crop_h"]

    abs_cx = crop_info["x1"] + cx * crop_w
    abs_cy = crop_info["y1"] + cy * crop_h
    abs_w = bw * crop_w
    abs_h = bh * crop_h

    x = abs_cx - abs_w * 0.5
    y = abs_cy - abs_h * 0.5

    img_h, img_w = image_shape[:2]

    x = np.clip(x, 0, img_w - 1)
    y = np.clip(y, 0, img_h - 1)
    abs_w = np.clip(abs_w, 1, img_w - x)
    abs_h = np.clip(abs_h, 1, img_h - y)

    return np.array([x, y, abs_w, abs_h], dtype=np.float32)


def load_checkpoint(model, checkpoint_path):
    state_dict = torch.load(checkpoint_path, map_location=device)

    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]

    state_dict = {
        key.replace("module.", "", 1): value
        for key, value in state_dict.items()
    }

    model.load_state_dict(state_dict, strict=True)


@torch.no_grad()
def track_sequence(seq_dir, model, transform):
    model.eval()

    gt_path = os.path.join(seq_dir, "groundtruth.txt")
    img_paths = sorted(glob.glob(os.path.join(seq_dir, "*.jpg")))

    if len(img_paths) == 0:
        raise RuntimeError(f"No JPG frames found in {seq_dir}")

    init_bbox = np.loadtxt(gt_path, delimiter=",", dtype=np.float32)
    if init_bbox.ndim > 1:
        init_bbox = init_bbox[0]

    first_img = cv2.imread(img_paths[0])
    if first_img is None:
        raise RuntimeError(f"Failed to read first frame in {seq_dir}")

    first_img = cv2.cvtColor(first_img, cv2.COLOR_BGR2RGB)

    template_crop, _ = crop_search_region(
        first_img,
        init_bbox,
        output_size=128,
        context_amount=0.5,
    )

    template = transform(template_crop).unsqueeze(0).to(device)

    state_bbox = init_bbox.astype(np.float32).copy()

    boxes = []
    times = []

    boxes.append([
        state_bbox[0] + 1.0,
        state_bbox[1] + 1.0,
        state_bbox[2],
        state_bbox[3],
    ])
    times.append(0.0)

    use_amp = device.type == "cuda"

    for frame_path in img_paths[1:]:
        start_time = time.perf_counter()

        image = cv2.imread(frame_path)

        if image is None:
            boxes.append([
                state_bbox[0] + 1.0,
                state_bbox[1] + 1.0,
                state_bbox[2],
                state_bbox[3],
            ])
            times.append(time.perf_counter() - start_time)
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        search_crop, crop_info = crop_search_region(
            image,
            state_bbox,
            output_size=256,
            context_amount=2.0,
        )

        search = transform(search_crop).unsqueeze(0).to(device)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=use_amp,
        ):
            pred_box, _, confidence = model(
                template,
                search,
                history_buffer=None,
            )

        pred_box = pred_box.squeeze(0).float().cpu().numpy()
        state_bbox = map_box_back(pred_box, crop_info, image.shape)

        boxes.append([
            state_bbox[0] + 1.0,
            state_bbox[1] + 1.0,
            state_bbox[2],
            state_bbox[3],
        ])
        times.append(time.perf_counter() - start_time)

    return np.asarray(boxes, dtype=np.float32), np.asarray(times, dtype=np.float32)


print("Device:", device)
print("Test path:", test_data_path)
print("Weights:", weights_path)

test_sequences = sorted(glob.glob(os.path.join(test_data_path, "GOT-10k_Test_*")))
print("Number of test sequences:", len(test_sequences))

if len(test_sequences) == 0:
    raise RuntimeError(f"No GOT-10K test sequences found at: {test_data_path}")

model = SpectreTrack(
    model_name="vit_base_patch16_224.mae",
    pretrained=False,
).to(device)

print("Loading checkpoint...")
load_checkpoint(model, weights_path)
model.eval()
print("Checkpoint loaded.")

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

tracker_dir = os.path.join(results_root, tracker_name)
os.makedirs(tracker_dir, exist_ok=True)

for seq_dir in tqdm(test_sequences, desc="Tracking GOT-10K test"):
    seq_name = os.path.basename(seq_dir)
    seq_result_dir = os.path.join(tracker_dir, seq_name)
    os.makedirs(seq_result_dir, exist_ok=True)

    all_times = []

    for rep in range(1, num_repetitions + 1):
        boxes, times = track_sequence(seq_dir, model, img_transform)
        all_times.append(times)

        box_file = os.path.join(seq_result_dir, f"{seq_name}_{rep:03d}.txt")
        np.savetxt(box_file, boxes, delimiter=",", fmt="%.3f")

    time_matrix = np.stack(all_times, axis=1)
    time_file = os.path.join(seq_result_dir, f"{seq_name}_time.txt")
    np.savetxt(time_file, time_matrix, delimiter=",", fmt="%.6f")

zip_path = f"{tracker_name}.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(tracker_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, results_root)
            zipf.write(file_path, arcname)

print("\nDone.")
print("Official GOT-10K folder:", os.path.abspath(tracker_dir))
print("Official GOT-10K zip:", os.path.abspath(zip_path))

Device: cuda:2
Test path: /root/victor/GOT-10K_FULL/test
Weights: spectre_got10k_mae_epoch_99.pth
Number of test sequences: 180
[INFO] Loading local checkpoint: /root/victor/vit_base_patch16_224.mae.pth
[INFO] Local checkpoint loaded successfully.
Loading checkpoint...
Checkpoint loaded.


Tracking GOT-10K test: 100%|██████████| 180/180 [23:39<00:00,  7.89s/it]


Done.
Official GOT-10K folder: /root/victor/results/SpectreTrack
Official GOT-10K zip: /root/victor/SpectreTrack.zip


sequence test

In [ ]:
import os
import glob
import cv2
import numpy as np

# --- 1. SET YOUR PATHS ---
val_data_path = '/root/victor/GOT-10K_FULL/val'
results_dir = 'evaluation_results'

# --- 2. CHOOSE THE SEQUENCE TO INSPECT ---
# Pick from the bottom of your list (e.g., 000020, 000055, 000137)
seq_name = 'GOT-10k_Val_000020'

seq_dir = os.path.join(val_data_path, seq_name)
pred_file = os.path.join(results_dir, f"{seq_name}.txt")
gt_file = os.path.join(seq_dir, 'groundtruth.txt')

if not os.path.exists(seq_dir):
    print(f"Directory not found: {seq_dir}")
    exit()

# Load images
img_paths = sorted(glob.glob(os.path.join(seq_dir, '*.jpg')))

# Load bounding boxes
try:
    gt_bboxes = np.loadtxt(gt_file, delimiter=',')
    if gt_bboxes.ndim == 1: gt_bboxes = gt_bboxes.reshape(1, -1)
except Exception as e:
    print(f"Error loading Ground Truth: {e}")
    exit()

try:
    pred_bboxes = np.loadtxt(pred_file, delimiter=',')
    if pred_bboxes.ndim == 1: pred_bboxes = pred_bboxes.reshape(1, -1)
except FileNotFoundError:
    print(f"Prediction file not found for {seq_name}. Did you run the evaluation script?")
    exit()

print(f"\n======================================")
print(f"Inspecting: {seq_name}")
print(f"======================================")
print("Controls:")
print("  [Space] - Pause / Play")
print("  [ n ]   - Step forward one frame (when paused)")
print("  [Esc]   - Quit viewer")
print(f"======================================\n")

paused = False
for i, img_path in enumerate(img_paths):
    img = cv2.imread(img_path)
    if img is None: continue

    # Draw Ground Truth (GREEN)
    if i < len(gt_bboxes) and not np.isnan(gt_bboxes[i][0]):
        gx, gy, gw, gh = map(int, gt_bboxes[i])
        cv2.rectangle(img, (gx, gy), (gx+gw, gy+gh), (0, 255, 0), 2)
        cv2.putText(img, "GT", (gx, gy-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Draw Prediction (RED)
    if i < len(pred_bboxes) and not np.isnan(pred_bboxes[i][0]):
        px, py, pw, ph = map(int, pred_bboxes[i])
        cv2.rectangle(img, (px, py), (px+pw, py+ph), (0, 0, 255), 2)
        cv2.putText(img, "Pred", (px, py-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Info overlay (Black background for readable text)
    overlay_text = f"Frame: {i}/{len(img_paths)} | [Space]: Pause/Play"
    cv2.rectangle(img, (5, 5), (400, 40), (0, 0, 0), -1)
    cv2.putText(img, overlay_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Tracking Inspector", img)

    # Playback Control Logic
    while True:
        delay = 0 if paused else 30  # 30ms for continuous playback, 0 waits indefinitely
        key = cv2.waitKey(delay) & 0xFF

        if key == 27:  # Esc
            cv2.destroyAllWindows()
            exit()
        elif key == ord(' '):  # Spacebar toggles pause
            paused = not paused
            if not paused: break
        elif key == ord('n') and paused:  # Step one frame
            break
        elif not paused:
            break

cv2.destroyAllWindows()